In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON


def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_region(data_folder: str, **storage_options) -> pd.DataFrame:
    # use memory mapping for faster I/O and inline path construction to eliminate temporaries
    return pd.read_csv(
        f"{data_folder}/region.tbl",
        sep='|',
        memory_map=True,
        **storage_options
    )

region = load_region(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_supplier(data_folder: str, **storage_options) -> pd.DataFrame:
    # build the file path and directly return the parsed DataFrame
    path = f"{data_folder}/supplier.tbl"
    return pd.read_csv(path, sep="|", memory_map=True, **storage_options)

supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# Filter and select only necessary columns, then chain merges to minimize intermediate copies and memory usage
date1 = pd.Timestamp('1996-01-01')
date2 = pd.Timestamp('1997-01-01')

# Pipeline: region → nation → customer → orders → lineitem → supplier
total = (
    region
    .loc[region.R_NAME == 'ASIA', ['R_REGIONKEY']]
    .merge(
        nation[['N_REGIONKEY', 'N_NATIONKEY', 'N_NAME']],
        left_on='R_REGIONKEY', right_on='N_REGIONKEY'
    )
    .merge(
        customer[['C_CUSTKEY', 'C_NATIONKEY']],
        left_on='N_NATIONKEY', right_on='C_NATIONKEY'
    )
    .merge(
        orders.loc[
            (orders.O_ORDERDATE >= date1) & (orders.O_ORDERDATE < date2),
            ['O_ORDERKEY', 'O_CUSTKEY']
        ],
        left_on='C_CUSTKEY', right_on='O_CUSTKEY'
    )
    .merge(
        lineitem[['L_ORDERKEY', 'L_SUPPKEY', 'L_EXTENDEDPRICE', 'L_DISCOUNT']],
        left_on='O_ORDERKEY', right_on='L_ORDERKEY'
    )
    .merge(
        supplier[['S_SUPPKEY', 'S_NATIONKEY']],
        left_on=['L_SUPPKEY', 'N_NATIONKEY'],
        right_on=['S_SUPPKEY', 'S_NATIONKEY']
    )
    .assign(TMP=lambda df: df.L_EXTENDEDPRICE * (1.0 - df.L_DISCOUNT))
    .groupby('N_NAME', as_index=False, sort=False)['TMP'].sum()
    .sort_values('TMP', ascending=False)
)